In [23]:
!pip install sentence-transformers chromadb groq pdfplumber

In [24]:
code = '''
import hashlib, logging, os, re, time
from dataclasses import dataclass
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

logging.basicConfig(level=logging.INFO)

@dataclass
class Chunk:
    text: str
    source: str
    chunk_index: int
    doc_id: str

    @property
    def chunk_id(self):
        return hashlib.md5((self.doc_id + "::" + str(self.chunk_index)).encode()).hexdigest()

@dataclass
class RetrievalResult:
    chunk: Chunk
    score: float

@dataclass
class RAGResponse:
    answer: str
    sources: list
    query: str
    latency_ms: float

class DocumentPreprocessor:
    def __init__(self, chunk_size=512):
        self.chunk_size = chunk_size

    def ingest(self, text, source):
        text = re.sub(r"\\s+", " ", text).strip()
        if not text:
            raise ValueError("Document is empty.")
        sentences = re.split(r"(?<=[.!?])\\s+", text)
        sentences = [s.strip() for s in sentences if s.strip()]
        chunks, current, current_len = [], [], 0
        for sent in sentences:
            if current_len + len(sent) > self.chunk_size and current:
                chunks.append(" ".join(current))
                current, current_len = [], 0
            current.append(sent)
            current_len += len(sent)
        if current:
            chunks.append(" ".join(current))
        doc_id = hashlib.md5(source.encode()).hexdigest()
        return [Chunk(text=c, source=source, chunk_index=i, doc_id=doc_id) for i, c in enumerate(chunks)]

class EmbeddingModel:
    def __init__(self):
        self._model = SentenceTransformer("all-MiniLM-L6-v2")

    def embed(self, texts):
        if not texts:
            return []
        return self._model.encode(texts, normalize_embeddings=True).tolist()

    def embed_one(self, text):
        return self.embed([text])[0]

class VectorStore:
    def __init__(self, persist_dir="./chroma_db"):
        self._client = chromadb.PersistentClient(path=persist_dir)
        self._col = self._client.get_or_create_collection(
            name="rag_documents",
            metadata={"hnsw:space": "cosine"}
        )

    def add_chunks(self, chunks, embeddings):
        ids = [c.chunk_id for c in chunks]
        docs = [c.text for c in chunks]
        metas = [{"source": c.source, "chunk_index": c.chunk_index, "doc_id": c.doc_id} for c in chunks]
        self._col.upsert(ids=ids, embeddings=embeddings, documents=docs, metadatas=metas)
        return len(ids)

    def query(self, query_embedding, top_k=5):
        count = self._col.count()
        if count == 0:
            return []
        results = self._col.query(
            query_embeddings=[query_embedding],
            n_results=min(top_k, count),
            include=["documents", "metadatas", "distances"]
        )
        output = []
        for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
            chunk = Chunk(text=doc, source=meta["source"], chunk_index=meta["chunk_index"], doc_id=meta["doc_id"])
            output.append(RetrievalResult(chunk=chunk, score=round(1.0 - dist, 4)))
        return output

    def delete_source(self, source):
        results = self._col.get(where={"source": source})
        ids = results.get("ids", [])
        if ids:
            self._col.delete(ids=ids)
        return len(ids)

    def list_sources(self):
        all_meta = self._col.get(include=["metadatas"])["metadatas"]
        return sorted(set(m["source"] for m in all_meta))

    @property
    def count(self):
        return self._col.count()

class LLMGenerator:
    def __init__(self, api_key=None):
        self._client = Groq(api_key=api_key or os.environ.get("GROQ_API_KEY"))

    def generate(self, query, context_chunks):
        if not context_chunks:
            return "I could not find any relevant context to answer your question."
        context = ""
        for i, r in enumerate(context_chunks, 1):
            context += "[" + str(i) + "] Source: " + r.chunk.source + "\\n" + r.chunk.text + "\\n\\n"
        prompt = "Answer using ONLY the context below. If not enough info, say so.\\n\\nContext:\\n" + context + "\\nQuestion: " + query
        response = self._client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

class RAGPipeline:
    def __init__(self, persist_dir="./chroma_db", top_k=5, min_score=0.25, api_key=None):
        self.top_k = top_k
        self.min_score = min_score
        self.preprocessor = DocumentPreprocessor()
        self.embedder = EmbeddingModel()
        self.vector_store = VectorStore(persist_dir=persist_dir)
        self.generator = LLMGenerator(api_key=api_key)

    def ingest_text(self, text, source):
        chunks = self.preprocessor.ingest(text, source)
        embeddings = self.embedder.embed([c.text for c in chunks])
        n = self.vector_store.add_chunks(chunks, embeddings)
        print("Ingested", n, "chunks from", source)
        return {"source": source, "chunks_stored": n}

    def query(self, question):
        if not question.strip():
            raise ValueError("Question cannot be empty.")
        t0 = time.perf_counter()
        q_emb = self.embedder.embed_one(question)
        raw = self.vector_store.query(q_emb, top_k=self.top_k)
        retrieved = [r for r in raw if r.score >= self.min_score]
        answer = self.generator.generate(question, retrieved)
        latency = (time.perf_counter() - t0) * 1000
        return RAGResponse(answer=answer, sources=retrieved, query=question, latency_ms=latency)

    def list_sources(self):
        return self.vector_store.list_sources()

    def delete_source(self, source):
        return self.vector_store.delete_source(source)

    @property
    def document_count(self):
        return self.vector_store.count
'''

with open("/content/rag_pipeline.py", "w") as f:
    f.write(code)

print("File created!")

File created!


In [25]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

print("Environment secured and API Key loaded.")

Environment secured and API Key loaded.


In [26]:
import sys
if "rag_pipeline" in sys.modules:
    del sys.modules["rag_pipeline"]

code = open("/content/rag_pipeline.py").read()
code = code.replace("llama3-8b-8192", "llama-3.1-8b-instant")
open("/content/rag_pipeline.py", "w").write(code)
print("Fixed!")

Fixed!


In [27]:
import sys
if "rag_pipeline" in sys.modules:
    del sys.modules["rag_pipeline"]

from rag_pipeline import RAGPipeline
pipe = RAGPipeline(api_key=os.environ["GROQ_API_KEY"])
print("Pipeline ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline ready!


In [28]:
pipe.ingest_text("""
Artificial Intelligence is the simulation of human intelligence in machines.
Machine learning is a subset of AI where systems learn from data.
Deep learning uses neural networks with many layers to learn complex patterns.
RAG stands for Retrieval Augmented Generation. It helps LLMs answer questions
using your own documents instead of just their training data.
""", source="ai_basics.txt")

Ingested 1 chunks from ai_basics.txt


{'source': 'ai_basics.txt', 'chunks_stored': 1}

In [29]:
response = pipe.query("What is machine learning?")
print("ANSWER:", response.answer)
print("Chunks used:", len(response.sources))
print("Time taken:", round(response.latency_ms), "ms")

ANSWER: Machine learning is a subset of AI where systems learn from data.
Chunks used: 1
Time taken: 377 ms


In [30]:
questions = [
    "What is Artificial Intelligence?",
    "What is deep learning?",
    "What is RAG and why is it useful?",
    "How are Large Language Models trained?",
]

for q in questions:
    r = pipe.query(q)
    print("Q:", q)
    print("A:", r.answer)
    print("Chunks used:", len(r.sources), "| Time:", round(r.latency_ms), "ms")
    print("---")

Q: What is Artificial Intelligence?
A: Artificial Intelligence is the simulation of human intelligence in machines.
Chunks used: 1 | Time: 487 ms
---
Q: What is deep learning?
A: Deep learning uses neural networks with many layers to learn complex patterns.
Chunks used: 1 | Time: 571 ms
---
Q: What is RAG and why is it useful?
A: RAG (Retrieval Augmented Generation) helps LLMs (Large Language Models) answer questions using your own documents instead of just their training data.
Chunks used: 1 | Time: 404 ms
---
Q: How are Large Language Models trained?
A: Not enough information in the given context to answer the question.
Chunks used: 1 | Time: 442 ms
---


In [31]:
print("=" * 50)
print("RAG PIPELINE - FINAL DEMO")
print("=" * 50)

print("\n INGESTED DOCUMENTS:")
for source in pipe.list_sources():
    print(" -", source)

print("\n TOTAL CHUNKS IN DATABASE:", pipe.document_count)

print("\n QUESTION & ANSWER DEMO:")
print("-" * 50)

questions = [
    "What is Artificial Intelligence?",
    "What is machine learning?",
    "What is deep learning?",
    "What is RAG and why is it useful?",
]

for q in questions:
    r = pipe.query(q)
    print("Q:", q)
    print("A:", r.answer)
    print("⏱ Time:", round(r.latency_ms), "ms |  Chunks used:", len(r.sources))
    print("-" * 50)

print("\n RAG PIPELINE WORKING SUCCESSFULLY!")

RAG PIPELINE - FINAL DEMO

 INGESTED DOCUMENTS:
 - ai_basics.txt

 TOTAL CHUNKS IN DATABASE: 1

 QUESTION & ANSWER DEMO:
--------------------------------------------------
Q: What is Artificial Intelligence?
A: Artificial Intelligence is the simulation of human intelligence in machines.
⏱ Time: 468 ms |  Chunks used: 1
--------------------------------------------------
Q: What is machine learning?
A: Machine learning is a subset of AI where systems learn from data.
⏱ Time: 385 ms |  Chunks used: 1
--------------------------------------------------
Q: What is deep learning?
A: Deep learning uses neural networks with many layers to learn complex patterns.
⏱ Time: 331 ms |  Chunks used: 1
--------------------------------------------------
Q: What is RAG and why is it useful?
A: RAG stands for Retrieval Augmented Generation. It helps LLMs (Large Language Models) answer questions using your own documents instead of just their training data.
⏱ Time: 273 ms |  Chunks used: 1
-----------------

In [37]:
# RAG PIPELINE - ARCHITECTURE EXPLANATION

print("""
HOW THIS RAG PIPELINE WORKS
============================

STEP 1 - DOCUMENT INGESTION:
  - Raw text is cleaned (whitespace removed)
  - Split into sentences
  - Grouped into chunks of 512 characters

STEP 2 - EMBEDDING GENERATION:
  - Each chunk is converted into a 384-dimension vector
  - Using sentence-transformers: all-MiniLM-L6-v2
  - Vectors are normalized for cosine similarity

STEP 3 - VECTOR STORAGE:
  - Chunks + embeddings stored in ChromaDB
  - Cosine similarity space for accurate retrieval
  - Persistent storage - survives session restart

STEP 4 - QUERY & RETRIEVAL:
  - User question is embedded using same model
  - Top-5 most similar chunks are retrieved
  - Only chunks with score >= 0.25 are used

STEP 5 - RESPONSE GENERATION:
  - Retrieved chunks sent to Llama 3.1 via Groq
  - LLM answers ONLY from provided context
  - Prevents hallucination

TECHNOLOGIES USED:
  - Embeddings : sentence-transformers (all-MiniLM-L6-v2)
  - Vector DB  : ChromaDB
  - LLM        : Llama 3.1 8B via Groq API
  - Language   : Python
""")


HOW THIS RAG PIPELINE WORKS

STEP 1 - DOCUMENT INGESTION:
  - Raw text is cleaned (whitespace removed)
  - Split into sentences
  - Grouped into chunks of 512 characters

STEP 2 - EMBEDDING GENERATION:
  - Each chunk is converted into a 384-dimension vector
  - Using sentence-transformers: all-MiniLM-L6-v2
  - Vectors are normalized for cosine similarity

STEP 3 - VECTOR STORAGE:
  - Chunks + embeddings stored in ChromaDB
  - Cosine similarity space for accurate retrieval
  - Persistent storage - survives session restart

STEP 4 - QUERY & RETRIEVAL:
  - User question is embedded using same model
  - Top-5 most similar chunks are retrieved
  - Only chunks with score >= 0.25 are used

STEP 5 - RESPONSE GENERATION:
  - Retrieved chunks sent to Llama 3.1 via Groq
  - LLM answers ONLY from provided context
  - Prevents hallucination

TECHNOLOGIES USED:
  - Embeddings : sentence-transformers (all-MiniLM-L6-v2)
  - Vector DB  : ChromaDB
  - LLM        : Llama 3.1 8B via Groq API
  - Language

In [36]:
# EDGE CASE HANDLING

print("EDGE CASE 1 - Empty question:")
try:
    pipe.query("   ")
except ValueError as e:
    print(" Caught:", e)

print("\nEDGE CASE 2 - Question with no relevant context:")
r = pipe.query("What is the price of gold in 2025?")
print(" Answer:", r.answer)

print("\nEDGE CASE 3 - List all sources in database:")
print(" Sources:", pipe.list_sources())

print("\nEDGE CASE 4 - Total chunks stored:")
print(" Total chunks:", pipe.document_count)

print("\n All edge cases handled correctly!")

EDGE CASE 1 - Empty question:
 Caught: Question cannot be empty.

EDGE CASE 2 - Question with no relevant context:
 Answer: I could not find any relevant context to answer your question.

EDGE CASE 3 - List all sources in database:
 Sources: ['ai_basics.txt']

EDGE CASE 4 - Total chunks stored:
 Total chunks: 1

 All edge cases handled correctly!
